In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print(torch.cuda.is_available())

True


In [5]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        #1 input image channel, 6 output channels, 5x5 square convolution kernel
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        #affine operation (Wx+b, linear transformation and translation. Scale and rotate by W, bias by b
        self.fc1 = nn.Linear(16*5*5, 120) #5x5 from image dimension
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # where do these dimensions come from?

    def forward(self, input):
        #convolution layer C1 (1 input image, 6 output
        #5x5 square convolution using RELU activation function, output Tensor with size (N, 6, 28, 28), where
        #N is size of batch (?) Where does 28, 28 come from?
        c1 = F.relu(self.conv1(input))
        #subsample layer s2, 2x2 grid, outputs (N, 6, 14, 14)
        s2 = F.max_pool2d(c1, (2,2))
        #Convolution layer c3: 6 input, 16 outputs, 5x5 square convolutions, RELU activation, outputs (N, 16, 10, 10). What is N?
        c3 = F.relu(self.conv2(s2))
        #subsample layer S4 2x2 grid, purely functional, no parameter, outputs (N, 16, 5, 5) Tensor
        s4 = F.max_pool2d(c3, 2)
        #flatten operation to output (N, 400)
        s4 = torch.flatten(s4, 1)
        #Fully connect layer 5: (N, 400) input, output (N, 120), RELU activation fn
        f5 = F.relu(self.fc1(s4))
        #fully connect F6 (N,120)->(N, 84), RELU fn
        f6 = F.relu(self.fc2(f5))
        #Fully connected output: (N, 84)->(N, 10)
        output = self.fc3(f6)
        return output

In [6]:
net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [9]:
params = list(net.parameters())
print(len(params))
print(params[0].size())

10
torch.Size([6, 1, 5, 5])


In [10]:
input = torch.randn(1, 1, 32, 32)
out = net(input)
print(out)

tensor([[-0.0047,  0.1011, -0.0112, -0.0249, -0.0146,  0.1469, -0.0266,  0.1024,
          0.0554, -0.0506]], grad_fn=<AddmmBackward0>)


In [12]:
net.zero_grad() #zero the gradient buffers
out.backward(torch.randn(1,10))

In [14]:
#Loss function
output = net(input)
target = torch.randn(10) #random target
target = target.view(1,-1) #match input's shape
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

tensor(1.2828, grad_fn=<MseLossBackward0>)


In [16]:
print(loss.grad_fn) #MSE Loss
print(loss.grad_fn.next_functions[0][0]) #linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0]) # ReLU

In [17]:
#clear existing gradients and backprop
net.zero_grad()
print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)

loss.backward()

print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
None
conv1.bias.grad after backward
tensor([ 0.0048,  0.0059, -0.0074, -0.0032,  0.0295,  0.0021])


In [20]:
print(net.conv1.weight.grad)

tensor([[[[-7.6142e-05, -1.6876e-02,  2.7218e-02, -2.5969e-03,  1.5381e-02],
          [ 1.4613e-02,  9.9854e-03, -8.2056e-03, -1.4564e-02, -8.5085e-03],
          [-2.3728e-02,  1.4608e-02,  9.5645e-03, -6.4885e-03, -9.1368e-03],
          [-5.0364e-04,  4.8465e-03, -2.0296e-03, -3.2696e-03,  8.4326e-03],
          [-5.9655e-03,  2.4317e-02,  1.0336e-02,  1.3945e-03,  2.9519e-03]]],


        [[[-3.9450e-03,  1.1588e-02, -3.3510e-03, -9.0798e-03, -1.7392e-02],
          [ 8.9701e-04,  3.2924e-02, -4.8900e-03,  2.3041e-02, -1.5435e-02],
          [ 1.0335e-02,  1.9516e-02, -4.1605e-03, -2.2289e-02,  1.9696e-02],
          [ 6.5347e-03, -9.2482e-03, -5.7824e-03,  3.1773e-02, -6.7924e-03],
          [ 1.1991e-03,  1.5383e-02, -2.2706e-02, -1.1259e-02,  1.3602e-02]]],


        [[[ 3.4239e-03, -4.5989e-03, -2.1917e-03, -6.3927e-03,  1.0991e-02],
          [-1.2882e-02,  9.1417e-04, -7.0662e-03,  1.2678e-03, -4.4794e-03],
          [-9.7788e-04,  1.1241e-02,  4.0845e-04,  6.3147e-03,  8.28

In [21]:
#updating weights: w = w-learning_rate*gradient
learning_rate = 0.01
for f in net.parameters():
    f.data.sub_(f.grad.data*learning_rate)

In [23]:
#alternative weight updates:
import torch.optim as optim

optimizer = optim.SGD(net.parameters(), lr=0.01)

#in training loop
optimizer.zero_grad()
output = net(input)
loss = criterion(output, target)
loss.backward()
optimizer.step()